# Module 1: Sentiment Analysis with Python
### Python Programming & AI — Hochschule für Wirtschaft und Umwelt (HfWU)

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Load a real-world dataset** from HuggingFace with a single line of code
2. **Explore and visualize** tweet data using `pandas` and `plotly`
3. **Clean and preprocess** raw social media text
4. **Transform text** into numerical features using TF-IDF
5. **Train and evaluate** a Naive Bayes sentiment classifier
6. **Test your model** on tweets you write yourself

---

## The Big Idea

> *"Every minute, hundreds of thousands of tweets are posted about brands, airlines, products, and politicians. No human team can read them all — but a machine can."*

**Sentiment Analysis** is one of the most widely deployed AI techniques in business today:
- Monitor brand reputation on social media in real time
- Automatically route negative customer feedback to support teams
- Track public reaction to product launches or news events

Today we'll use **45,000+ real tweets** to build a working sentiment classifier from scratch.

---

**Estimated Time:** 75–90 minutes  
**Dataset:** `tweet_eval` via HuggingFace — 45,000+ real tweets, 3 sentiment classes  
**Libraries:** `datasets`, `pandas`, `sklearn`, `plotly`, `re`

---

> **What is HuggingFace?**
> Think of it as the **"GitHub for AI"** — the world's largest open platform for AI models and datasets.
> Over 500,000 datasets are freely available. **No login, no API key, no cost.**

---
## Step 1: Install & Import Libraries

| Library | Purpose |
|---|---|
| `datasets` | Load real-world datasets from HuggingFace Hub |
| `pandas` | Tabular data manipulation |
| `sklearn` | ML algorithms (TF-IDF, Naive Bayes, metrics) |
| `plotly` | Interactive visualizations |
| `re` | Regular expressions for text cleaning |

In [17]:
# Install required libraries (run once — takes ~30 seconds)
!pip install datasets pandas scikit-learn plotly --quiet
print("Libraries ready!")

Libraries ready!


In [19]:
print('Attempting to fix HfUriError by installing specific library versions...')
!pip install datasets==2.14.5 huggingface_hub==0.16.4 --quiet
print('Specific library versions installed! Please run the `load-data-cell` now.')

Attempting to fix HfUriError by installing specific library versions...
Specific library versions installed! Please run the `load-data-cell` now.


In [20]:
import pandas as pd
import numpy as np
import re
import time
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully!")

All libraries imported successfully!


---
## Step 2: Load Real Twitter Data from HuggingFace

We use the **`tweet_eval`** dataset — a standard academic benchmark used in research papers worldwide.
The `sentiment` subset contains **~45,000 real tweets** with 3 labels:

| Label | Meaning | Example |
|---|---|---|
| `0` → negative | Bad experience | *"This airline lost my luggage AGAIN"* |
| `1` → neutral  | Factual / no emotion | *"Flight departs at 6pm from Terminal 2"* |
| `2` → positive | Good experience | *"Amazing crew, arrived early!"* |

> **One line of code. No login. No download. No API key.**
> This is how professional data scientists access datasets today.

In [21]:
print('Clearing Hugging Face cache and retrying...')
!rm -rf ~/.cache/huggingface/datasets

print("Downloading tweet_eval dataset from HuggingFace...")
# Explicitly using the 'cardiffnlp' namespace to satisfy HfUriError
raw_dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
print("Dataset loaded!\n")

print("Available splits:")
for split, data in raw_dataset.items():
    print(f"  {split:<12}: {len(data):>6,} tweets")

Clearing Hugging Face cache and retrying...


Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset loaded!

Available splits:
  train       : 45,615 tweets
  test        : 12,284 tweets
  validation  :  2,000 tweets


In [22]:
# ── Label mapping ────────────────────────────────────────────
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

# ── Training data: train + validation splits ──────────────────
# We combine train and validation here because:
#   - validation was used by the dataset authors to tune THEIR models
#   - we will do our own honest evaluation on the held-out test split
df_train = pd.concat([
    raw_dataset['train'].to_pandas(),
    raw_dataset['validation'].to_pandas()
], ignore_index=True)
df_train['sentiment'] = df_train['label'].map(label_map)

# ── Test data: held-out test split (never touches training) ───
df_test = raw_dataset['test'].to_pandas()
df_test['sentiment'] = df_test['label'].map(label_map)

# ── Combined df for EDA only ──────────────────────────────────
df = pd.concat([df_train, df_test], ignore_index=True)

print(f"Training pool (train + validation) : {len(df_train):,} tweets")
print(f"Test set (held-out, never touched) : {len(df_test):,} tweets")
print(f"\nSentiment distribution (full dataset):")
print(df['sentiment'].value_counts().to_string())
print()
df[['text', 'sentiment']].head(8)




Training pool (train + validation) : 47,615 tweets
Test set (held-out, never touched) : 12,284 tweets

Sentiment distribution (full dataset):
sentiment
neutral     27479
positive    21043
negative    11377



,text,sentiment
0,"""QT @user In the original draft of the 7th boo...",positive
1,"""Ben Smith / Smith (concussion) remains out of...",neutral
2,Sorry bout the stream last night I crashed out...,neutral
3,Chase Headley's RBI double in the 8th inning o...,neutral
4,@user Alciato: Bee will invest 150 million in ...,positive
5,@user LIT MY MUM 'Kerry the louboutins I wonde...,positive
6,"""\"""""""" SOUL TRAIN\"""""""" OCT 27 HALLOWEEN SPECIA...",positive
7,So disappointed in wwe summerslam! I want to s...,negative


In [23]:
df.to_excel('tweets_data.xlsx', index=False)

from google.colab import files
files.download('tweets_data.xlsx')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Step 3: Exploratory Data Analysis (EDA)

Before any modelling, we **explore the data**. Key questions:
- Is the dataset **balanced** across classes?
- How long are tweets by sentiment?
- What does the raw data actually look like?

EDA first, model second — this is non-negotiable professional practice.

In [25]:
df['tweet_length'] = df['text'].apply(lambda x: len(str(x).split()))

print("Tweet length by sentiment (words):")
print(df.groupby('sentiment')['tweet_length'].describe().round(1).to_string())

Tweet length by sentiment (words):
             count  mean  std  min   25%   50%   75%   max
sentiment                                                 
negative   11377.0  19.0  5.2  1.0  16.0  20.0  23.0  35.0
neutral    27479.0  17.9  5.4  1.0  14.0  18.0  22.0  33.0
positive   21043.0  18.5  5.3  1.0  15.0  19.0  22.0  34.0


In [26]:
dist = df['sentiment'].value_counts().reset_index()
dist.columns = ['sentiment', 'count']

fig1 = px.bar(
    dist, x='sentiment', y='count',
    color='sentiment',
    color_discrete_map={'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#3498db'},
    title='Class Distribution — How Many Tweets per Sentiment?',
    labels={'count': 'Number of Tweets', 'sentiment': 'Sentiment'},
    text='count'
)
fig1.update_traces(textposition='outside')
fig1.show()

# Discussion: the dataset is IMBALANCED.
# More neutral tweets than positive or negative.
# This is very common in real-world data and affects model performance.

In [27]:
fig2 = px.box(
    df, x='sentiment', y='tweet_length',
    color='sentiment',
    color_discrete_map={'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#3498db'},
    points='outliers',
    title='Tweet Length by Sentiment — Do Negative Tweets Run Longer?',
    labels={'tweet_length': 'Word Count', 'sentiment': 'Sentiment'}
)
fig2.show()

In [33]:
# Print 3 sample tweets per class to build intuition
print("Sample tweets from each class:\n")
for sentiment in ['positive', 'negative', 'neutral']:
    samples = df[df['sentiment'] == sentiment]['text'].sample(3, random_state=42).tolist()
    print(f"{'='*60}")
    print(f"  {sentiment.upper()}")
    print(f"{'='*60}")
    for s in samples:
        print(f"  -> {s}")
    print()

Sample tweets from each class:

  POSITIVE
  -> It was such a shame that the Great Yorkshire Show was cancelled but we still really enjoyed Tuesday and can't...
  -> My mom is really lecturing me about how Sharknado is such a good movie &amp; that everyone watches it which is why they came out with a 3rd one.
  -> I have the most amazing dreams... I was at the gym doing lungNsquats!  Justin woke me up on my 3rd set-.- lmao

  NEGATIVE
  -> "Tho I respect his decision, I don't agree with it. James McClean may not necessarily be a supporter of IRA. He said the poppys stand for ..."
  -> "Charlie Hebdo may quite rightly enjoy freedom of speech, but to mock the deaths of over 200 people? I think they have crossed a line"
  -> Jiujitsu test tomorrow... I think it might be the first time im nervous about... Anything in almost a year

  NEUTRAL
  -> Sore left knee keeping Hamilton out: Rangers outfielder Josh Hamilton was not in the lineup in Tuesday's 3-2 l...
  -> "Caitlyn Jenner and her ex

---
## Step 4: Text Preprocessing

Twitter text is particularly messy. We need to clean it before feeding it to a model:

| Problem | Example | Fix |
|---|---|---|
| URLs | `http://t.co/abc123` | Remove completely |
| Mentions | `@united @Delta` | Remove (username ≠ sentiment) |
| Hashtags | `#fail #delayed` | Keep word, remove `#` symbol |
| Uppercase | `"TERRIBLE SERVICE"` | Lowercase everything |
| Special chars | `"omg!!!!!"` | Remove non-alphanumeric |
| Extra spaces | `"  great  flight  "` | Strip and normalize |

In [34]:
def clean_tweet(text):
    """
    Cleans a tweet for NLP:
    removes URLs, mentions, hashtag symbols, punctuation;
    lowercases and normalizes whitespace.
    """
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)    # remove URLs
    text = re.sub(r'@\w+', '', text)                 # remove @mentions
    text = re.sub(r'#', '', text)                    # remove # symbol, keep word
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Preview: compare a raw tweet vs its cleaned version
sample_raw = df_train['text'].iloc[5]
print("BEFORE:", sample_raw)
print("\nAFTER: ", clean_tweet(sample_raw))
print("\nNote: cleaning is applied to both train and test sets in Step 5.")

BEFORE: @user LIT MY MUM 'Kerry the louboutins I wonder how many Willam owns!!! Look Kerry Warner Wednesday!'

AFTER:  lit my mum kerry the louboutins i wonder how many willam owns look kerry warner wednesday

Note: cleaning is applied to both train and test sets in Step 5.


---
## Step 4b: Feature Engineering

**Feature engineering** means creating new input variables from raw data that help the model learn better.
TF-IDF captures *which words* appear — but it misses signals like tone, intensity, and writing style.

We engineer four handcrafted features from the raw tweet text:

| Feature | What it captures | Example signal |
|---|---|---|
| `tweet_length` | Number of words | Negative tweets tend to be longer (more to complain about) |
| `has_exclamation` | Contains `!` | Strong emotion — both very positive AND very negative |
| `caps_ratio` | % uppercase characters | ALL CAPS often signals anger or excitement |
| `has_negation` | Contains *not, never, no, won't* | Flips meaning — *"not good"* is very different from *"good"* |

These will be **combined with TF-IDF** in Step 5 to give the model a richer picture of each tweet.

> This is what **feature engineering in AI workflows** looks like in practice:
> domain knowledge (what makes a tweet negative?) translated into measurable numerical signals.

In [35]:
import scipy.sparse as sp

NEGATION_WORDS = {'not', 'never', 'no', 'nobody', 'nothing', 'neither',
                  'nor', 'nowhere', 'hardly', 'barely', 'scarcely',
                  "won't", "can't", "don't", "didn't", "isn't", "wasn't"}

def engineer_features(text_series):
    """
    Takes a pandas Series of raw tweet strings.
    Returns a sparse matrix of 4 handcrafted features per tweet.
    """
    features = pd.DataFrame(index=text_series.index)

    # Feature 1: tweet length in words
    features['tweet_length'] = text_series.apply(
        lambda t: len(str(t).split())
    )

    # Feature 2: presence of exclamation mark (1 = yes, 0 = no)
    features['has_exclamation'] = text_series.apply(
        lambda t: int('!' in str(t))
    )

    # Feature 3: ratio of uppercase characters (0.0 to 1.0)
    features['caps_ratio'] = text_series.apply(
        lambda t: sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1)
    )

    # Feature 4: presence of negation words (1 = yes, 0 = no)
    features['has_negation'] = text_series.apply(
        lambda t: int(bool(set(str(t).lower().split()) & NEGATION_WORDS))
    )

    # Return as sparse matrix so it can be combined with TF-IDF later
    return sp.csr_matrix(features.values.astype(float))

# Quick preview on training data
sample_preview = df_train[['text']].head(10).copy()
sample_preview['tweet_length']    = df_train['text'].head(10).apply(lambda t: len(str(t).split()))
sample_preview['has_exclamation'] = df_train['text'].head(10).apply(lambda t: int('!' in str(t)))
sample_preview['caps_ratio']      = df_train['text'].head(10).apply(
    lambda t: round(sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1), 3)
)
sample_preview['has_negation']    = df_train['text'].head(10).apply(
    lambda t: int(bool(set(str(t).lower().split()) & NEGATION_WORDS))
)
print("Engineered features preview:")
sample_preview

Engineered features preview:


,text,tweet_length,has_exclamation,caps_ratio,has_negation
0,"""QT @user In the original draft of the 7th boo...",18,0,0.092,0
1,"""Ben Smith / Smith (concussion) remains out of...",14,0,0.119,0
2,Sorry bout the stream last night I crashed out...,24,0,0.032,0
3,Chase Headley's RBI double in the 8th inning o...,23,0,0.072,0
4,@user Alciato: Bee will invest 150 million in ...,21,0,0.044,0
5,@user LIT MY MUM 'Kerry the louboutins I wonde...,17,1,0.149,0
6,"""\"""""""" SOUL TRAIN\"""""""" OCT 27 HALLOWEEN SPECIA...",23,0,0.400,0
7,So disappointed in wwe summerslam! I want to s...,15,1,0.026,0
8,"""This is the last Sunday w/o football .....,NF...",11,0,0.082,0
9,@user @user CENA & AJ sitting in a tree K-I-S-...,24,0,0.150,0


---
## Step 5: Feature Extraction — TF-IDF

ML models understand numbers, not words. **TF-IDF** converts each tweet into a vector:

- **TF** (Term Frequency): How often does a word appear in *this* tweet?
- **IDF** (Inverse Document Frequency): How rare is this word across *all* tweets?

> Words like *"the"*, *"is"* → appear everywhere → low score → low importance  
> Words like *"delayed"*, *"amazing"*, *"refund"* → rare and distinctive → high score

### Data Leakage — The Most Common ML Mistake

We use the **original dataset splits** exactly as intended:

| Data | Used for | Size |
|---|---|---|
| `train` + `validation` | Model training | 47,615 tweets |
| `test` | Final evaluation only — **never seen during training** | 12,284 tweets |

> **Data leakage** happens when test data accidentally influences training.
> If we mixed all splits and then randomly re-split, test tweets could appear in training — giving
> falsely high accuracy. The model would have already "seen the exam answers".
> This is one of the most common and costly mistakes in real ML projects.

In [36]:
import re                                             # Regular expressions used to clean data


def clean_tweet(text):
    """
    Cleans a tweet for NLP:
    removes URLs, mentions, hashtag symbols, punctuation;
    lowercases and normalizes whitespace.
    """
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)    # remove URLs
    text = re.sub(r'@\w+', '', text)                 # remove @mentions
    text = re.sub(r'#', '', text)                    # remove # symbol, keep word
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Apply cleaning to both splits ────────────────────────────
df_train['cleaned_text'] = df_train['text'].apply(clean_tweet)
df_test['cleaned_text']  = df_test['text'].apply(clean_tweet)

# ── For classroom speed: sample from training pool ────────────
# Remove .sample() to train on the full 47,615 tweets
df_train_sample = df_train.sample(5000, random_state=42)

#Standard ML convention — X is input, y is the correct answer label we want the model to learn to predict.
X_train_text = df_train_sample['cleaned_text']   # input: what the model reads
X_train_raw  = df_train_sample['text']           # input: raw, for engineered features
y_train      = df_train_sample['sentiment']      # answer: positive/negative/neutral


X_test_text  = df_test['cleaned_text']            # input: cleaned test tweets
X_test_raw   = df_test['text']                   # input: raw test tweets, for engineered features
y_test       = df_test['sentiment']              # answer: true labels to compare predictions against


print(f"Training samples : {len(X_train_text):,}")
print(f"Test samples     : {len(X_test_text):,}  (original held-out test split)")
print("No data leakage — test tweets were never seen during training.\n")

# ── TF-IDF features ───────────────────────────────────────────
#learn the vocabulary AND convert to numbers.Transform only on test = convert using the already-learned vocabulary. Never re-learn on test data
vectorizer    = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=5, stop_words='english', max_df=0.7)
X_train_tfidf = vectorizer.fit_transform(X_train_text) # learns vocabulary from train
X_test_tfidf  = vectorizer.transform(X_test_text)        # applies same vocabulary to test

# ── Handcrafted features (from Step 4b) ───────────────────────
X_train_eng   = engineer_features(X_train_raw)
X_test_eng    = engineer_features(X_test_raw)

# ── Combine: TF-IDF + engineered features side by side (both features)
#Runs our engineer_features() function from Step 4b on the raw text. Returns a small matrix of 5,000 tweets × 4 features (length, exclamation, caps, negation).
# sp.hstack joins two sparse matrices horizontally
X_train_tfidf = sp.hstack([X_train_tfidf, X_train_eng])
X_test_tfidf  = sp.hstack([X_test_tfidf,  X_test_eng])

print(f"Combined feature matrix (train): {X_train_tfidf.shape[0]:,} tweets x {X_train_tfidf.shape[1]:,} features")
print(f"  └─ {X_train_tfidf.shape[1] - 4} TF-IDF features + 4 engineered features")

#summarizes
#Clean the tweets, convert them to numbers two different ways, then combine both number representations into one final input matrix for the model.

Training samples : 5,000
Test samples     : 12,284  (original held-out test split)
No data leakage — test tweets were never seen during training.

Combined feature matrix (train): 5,000 tweets x 2,197 features
  └─ 2193 TF-IDF features + 4 engineered features


---
## Step 6: Train & Evaluate — Naive Bayes

**Multinomial Naive Bayes** is the classic algorithm for text classification: fast, interpretable, strong baseline.

It calculates the probability that a tweet belongs to each class based on the words it contains:
```
P(negative | 'delayed', 'cancelled', 'terrible') ≈ very high
P(positive | 'amazing', 'love', 'great crew')    ≈ very high
```
Called "Naive" because it assumes all words are independent — oversimplified, but works well.

In [37]:
# ── Initialize the model with smoothing parameter alpha ───────
# alpha=0.5 (Laplace smoothing) prevents zero probabilities
# for words not seen during training
model = MultinomialNB(alpha=0.5)

# ── Train: model learns sentiment patterns from 5,000 tweets ──
# X_train_tfidf = 5,000 × 2,197 feature matrix (TF-IDF + engineered)
# y_train       = correct sentiment labels for those 5,000 tweets
model.fit(X_train_tfidf, y_train)

# ── Predict: run model on 12,284 unseen test tweets ───────────
# model has never seen these tweets — honest evaluation
y_pred = model.predict(X_test_tfidf)

# ── Overall accuracy: % of all 12,284 predictions correct ─────
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.1%}\n")

# ── Per-class breakdown: precision, recall, F1 per sentiment ──
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

Overall Accuracy: 50.7%

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0.60      0.14      0.23      3972
     neutral       0.54      0.72      0.62      5937
    positive       0.40      0.58      0.47      2375

    accuracy                           0.51     12284
   macro avg       0.51      0.48      0.44     12284
weighted avg       0.53      0.51      0.46     12284



### Understanding the Results

| What you see | What it means |
|---|---|
| **50.7% accuracy** | Model gets 1 in 2 tweets right — weak, but expected: only 5,000 training tweets vs 12,284 test tweets |
| **Neutral F1 = 0.62** | Best performing class — makes sense, it has the most training examples (class imbalance from Step 3) |
| **Negative F1 = 0.23** | Worst — model almost never correctly identifies negative tweets (recall only 14%) |
| **Support 3972 / 5937 / 2375** | How many real test tweets exist per class — neutral dominates, which biases the model |

> **Try Exercise 2** — train on the full 47,615 tweets instead of 5,000 and watch accuracy jump significantly.

### Understanding the Metrics

| Metric | What it means |
|---|---|
| **Accuracy** | % of all predictions correct |
| **Precision** | Of predicted positives, how many were actually positive? |
| **Recall** | Of all actual positives, how many did we catch? |
| **F1-Score** | Harmonic mean of precision & recall — best single metric |
| **Support** | Number of real samples for that class in the test set |

> **Discussion question:** Why does 'negative' have the lowest F1 here, and 'neutral' the highest?
> *(Hint: compare the class distribution chart from Step 3 with the confusion matrix below)*

In [39]:
labels = ['negative', 'neutral', 'positive']
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=labels, y=labels,
    title='Confusion Matrix — Where Does the Model Get Confused?',
    color_continuous_scale='Blues',
    text_auto=True
)
fig_cm.update_layout(width=500, height=450)
fig_cm.show()
# Diagonal = correct predictions
# Off-diagonal = mistakes — which class gets confused with which?

### Understanding the Confusion Matrix

The diagonal (top-left to bottom-right) = **correct predictions**.
Everything off-diagonal = **mistakes**.

| Actual → Predicted | What happened | Count |
|---|---|---|
| negative → **negative** | Correctly identified negative tweets | 558 |
| negative → neutral  | Negative tweets mistaken as neutral | 2654 |
| negative → positive | Negative tweets mistaken as positive | 760 |
| neutral → negative  | Neutral tweets mistaken as negative | 335 |
| neutral → **neutral**  | Correctly identified neutral tweets | 4286 |
| neutral → positive | Neutral tweets mistaken as positive | 1316 |
| positive → negative | Positive tweets mistaken as negative | 44 |
| positive → neutral | Positive tweets mistaken as neutral | 950 |
| positive → **positive**  | Correctly identified positive tweets | 1381 |

### Key Insights

- **Neutral is the strongest class** — 4,286 correct out of 5,937 (72% recall)
- **Negative is the weakest class** — only 558 correct out of 3,972 (14% recall).
  The model almost always misreads negative tweets as neutral instead.
- **The model is biased toward neutral** — when unsure, it defaults to neutral
  because neutral tweets dominate the training data (class imbalance).

> **Discussion question for class:** Why do you think negative tweets
> are the hardest to detect? What makes a tweet "clearly negative"
> vs. something a model might read as neutral?

---
## Step 6b: AI Workflow Integration — sklearn Pipeline

So far we've run each step manually: clean → vectorize → engineer features → train → predict.
In production AI systems, these steps are **chained into a single Pipeline object**.

A **Pipeline** is a reusable, deployable workflow:

```
Raw Tweet Text
      ↓
  TF-IDF Vectorizer       ← learns vocabulary from training data
      ↓
  Naive Bayes Classifier  ← learns sentiment patterns
      ↓
  Prediction: positive / negative / neutral
```

**Why this matters in real AI workflows:**
- One `pipeline.fit()` call trains the entire chain
- One `pipeline.predict()` call processes raw text end-to-end
- No risk of applying steps in the wrong order
- Can be saved to disk and deployed as a service (FastAPI, Flask, etc.)

> This is exactly the pattern used in production ML systems at companies like Google, Amazon, and Deutsche Bahn.

Note: the pipeline below uses TF-IDF only. It leaves out the four handcrafted features from Step 4b
and uses slightly different TF-IDF settings, so its accuracy is close to the manual result
but not identical.


In [40]:
from sklearn.pipeline import Pipeline

# ── Build the pipeline ────────────────────────────────────────
# Each tuple is: ('step_name', TransformerOrClassifier)
# Steps execute in order: input → tfidf → model → output
sentiment_pipeline = Pipeline([
    ('tfidf',  TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)),
    ('model',  MultinomialNB(alpha=0.5))
])

# ── Train the entire pipeline in one call ─────────────────────
# No separate vectorizer.fit_transform() needed — the pipeline handles it
sentiment_pipeline.fit(X_train_text, y_train)

# ── Evaluate ──────────────────────────────────────────────────
y_pred_pipeline = sentiment_pipeline.predict(X_test_text)
pipeline_acc    = accuracy_score(y_test, y_pred_pipeline)

print(f"Pipeline accuracy: {pipeline_acc:.1%}")
print(f"(Same as manual approach — pipeline just packages the steps cleanly)\n")
print(sentiment_pipeline)

Pipeline accuracy: 50.0%
(Same as manual approach — pipeline just packages the steps cleanly)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=3000, min_df=2,
                                 ngram_range=(1, 2))),
                ('model', MultinomialNB(alpha=0.5))])


In [41]:
# ── End-to-end prediction: raw text in, label out ─────────────
# This is what a deployed API endpoint would look like

test_tweets = [
    "Incredible flight, crew was so friendly and we arrived early!",
    "Flight delayed 4 hours, nobody answered at customer service.",
    "Flight was on time but the food was disappointing."
]

# Raw text goes straight in — no manual cleaning needed
predictions = sentiment_pipeline.predict(test_tweets)
probabilities = sentiment_pipeline.predict_proba(test_tweets)

print("End-to-end pipeline predictions:\n")
print(f"{'Tweet':<55} {'Prediction':<12} {'Confidence'}")
print("-" * 80)
for tweet, pred, probs in zip(test_tweets, predictions, probabilities):
    conf = max(probs)
    print(f"{tweet[:52]+'...':<55} {pred:<12} {conf:.0%}")

End-to-end pipeline predictions:

Tweet                                                   Prediction   Confidence
--------------------------------------------------------------------------------
Incredible flight, crew was so friendly and we arriv... positive     59%
Flight delayed 4 hours, nobody answered at customer ... positive     47%
Flight was on time but the food was disappointing....   neutral      43%


In [42]:
# ── Save the pipeline to disk ─────────────────────────────────
# A saved pipeline can be loaded in any Python app — a web server,
# a FastAPI endpoint, a scheduled batch job, etc.
import joblib

joblib.dump(sentiment_pipeline, 'sentiment_pipeline.pkl')
print("Pipeline saved to: sentiment_pipeline.pkl")

# ── Reload and use ────────────────────────────────────────────
loaded_pipeline = joblib.load('sentiment_pipeline.pkl')
result = loaded_pipeline.predict(["Best airline experience I have ever had!"])
print(f"Loaded pipeline prediction: {result[0]}")
print("\nThis is how ML models are deployed in production systems.")

Pipeline saved to: sentiment_pipeline.pkl
Loaded pipeline prediction: positive

This is how ML models are deployed in production systems.


## How AI Works in This Notebook — Simply Explained

---

### The AI learns in 3 stages:

**Stage 1 — Show examples (Training)**
We feed 5,000 tweets with correct labels to the model.
The AI reads them all and counts which words appear in which sentiment.
The numbers below are made up to show the idea, they are not taken from this dataset.

| Word | NEGATIVE | NEUTRAL | POSITIVE |
|---|---|---|---|
| `"terrible"` | 890× | 8× | 3× |
| `"amazing"` | 4× | 15× | 810× |
| `"delayed"` | 760× | 21× | 3× |

> No human told it any of this — it figured it out by counting patterns.

---

**Stage 2 — Make decisions (Prediction)**
A new tweet arrives. The AI looks up each word in its table,
multiplies the probabilities, and picks the highest score.
"The flight was delayed and terrible"
→ "delayed" = 76% NEGATIVE
→ "terrible" = 89% NEGATIVE
→ Final answer: NEGATIVE

---

**Stage 3 — Check performance (Evaluation)**
We test it on 12,284 tweets it has never seen.
We compare predictions vs. correct answers → **50.7% accuracy**.

---

> **Bottom line:** There is no LLM, no API, no internet.
> The AI is a fast pattern-counting machine that learned from
> examples — like a student studying past exam papers.

---
## Step 7: What Did the Model Learn?

A key advantage of Naive Bayes over deep neural networks: full **interpretability**.
We can see exactly which words drive each sentiment prediction.

In [43]:
feature_names = list(vectorizer.get_feature_names_out()) + \
                ['tweet_length', 'has_exclamation', 'caps_ratio', 'has_negation']
top_n = 12
rows = []

for i, cls in enumerate(model.classes_):
    top_idx = model.feature_log_prob_[i].argsort()[-top_n:][::-1]
    for j in top_idx:
        rows.append({
            'word': feature_names[j],
            'score': model.feature_log_prob_[i][j],
            'sentiment': cls
        })

words_df = pd.DataFrame(rows)

fig_words = px.bar(
    words_df, x='score', y='word',
    color='sentiment', facet_col='sentiment',
    color_discrete_map={'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#3498db'},
    title='Top Predictive Words per Sentiment Class',
    labels={'score': 'Log Probability', 'word': 'Word / Phrase'},
    orientation='h', height=520
)
fig_words.update_yaxes(matches=None)
fig_words.show()

## Understanding: Top Predictive Words per Sentiment Class

This chart lists the 12 features with the highest score in each class.

**How to read it:**
- Each bar is one word or one handcrafted feature
- A bar closer to 0 means the feature carries more weight in that class
- The model ranked these automatically

---

### What the chart shows

**1. The four handcrafted features come first in every class**

`tweet_length`, `has_exclamation`, `has_negation` and `caps_ratio` sit at the top of all three panels.
This does not mean they are the best predictors. Naive Bayes scores a feature by how much of the
total weight in a class it collects. A tweet length of 18 is a much larger number than a TF-IDF
value (always below 1), and the yes/no features are switched on in a large share of all tweets.
So these four collect a lot of weight in every class, whatever the sentiment.

> **Lesson:** when you mix features with different scales, a ranking like this mostly reflects scale.
> In Module 2 we use `StandardScaler` to put features on a common scale.

**2. Most of the words are the same in every class**

Stop words are already removed (`stop_words='english'` in Step 5), but `tomorrow`, `just` and `going`
still appear in all three panels. They are frequent everywhere, so they tell us little about sentiment.

**3. A few words do differ**

`good` and `happy` appear only in the positive panel. `dont`, `like` and `think` appear only in the
negative panel. These are the kind of words that actually separate the classes.

---

### What to try next

A better question than "which words are most frequent in a class?" is
"which words are much more frequent in one class than in the others?".
Subtract the scores of two classes (for example positive minus negative) and plot the largest
differences. Sentiment words should then move to the top.


---
## Step 8: Test It on Your Own Tweets!

The model was trained on 5,000 real tweets. Let's see how it handles sentences we type ourselves.

In [44]:
def predict_sentiment(text):
    # Step 1: clean the text
    cleaned = clean_tweet(text)

    # Step 2: TF-IDF features
    tfidf_features = vectorizer.transform([cleaned])

    # Step 3: engineered features (must match what model was trained on)
    eng_features = engineer_features(pd.Series([text]))

    # Step 4: combine both — same as we did in Step 5
    vectorized = sp.hstack([tfidf_features, eng_features])

    # Step 5: predict
    prediction    = model.predict(vectorized)[0]
    probabilities = model.predict_proba(vectorized)[0]
    class_labels  = list(model.classes_)

    emoji_map = {'positive': 'POSITIVE', 'negative': 'NEGATIVE', 'neutral': 'NEUTRAL'}
    print(f'\nInput      : "{text}"')
    print(f'Prediction : {emoji_map[prediction]}')
    print(f'Confidence : {max(probabilities):.0%}\n')
    for label, prob in sorted(zip(class_labels, probabilities), key=lambda x: -x[1]):
        bar = 'X' * int(prob * 30)
        print(f'  {label:<10}: {bar:<30} {prob:.0%}')
    print()

predict_sentiment("Incredible flight, crew was so friendly and we arrived early!")


Input      : "Incredible flight, crew was so friendly and we arrived early!"
Prediction : POSITIVE
Confidence : 83%

  positive  : XXXXXXXXXXXXXXXXXXXXXXXX       83%
  neutral   : XX                             8%
  negative  : XX                             8%



In [48]:
predict_sentiment("Flight is comfortable.")


Input      : "Flight is comfortable."
Prediction : POSITIVE
Confidence : 44%

  positive  : XXXXXXXXXXXXX                  44%
  neutral   : XXXXXXXXX                      31%
  negative  : XXXXXXX                        25%



In [50]:
predict_sentiment("Flight was on time.")


Input      : "Flight was on time."
Prediction : POSITIVE
Confidence : 46%

  positive  : XXXXXXXXXXXXX                  46%
  neutral   : XXXXXXXXX                      33%
  negative  : XXXXXX                         20%



In [ ]:
# Write your own tweet and see what the model predicts!
predict_sentiment("Write your own tweet here and see what happens.")


Input      : "Write your own tweet here and see what happens."
Prediction : NEUTRAL
Confidence : 62%

  neutral   : XXXXXXXXXXXXXXXXXX             62%
  positive  : XXXXXXXX                       29%
  negative  : XX                             9%



---
## YOUR TURN — Exercises

---

### Exercise 1 — Binary vs. Three-Class Problem

Our model predicts 3 classes. Simplify to a binary problem by removing neutral tweets and retraining.
Does accuracy improve? Why or why not?

> Hint: `df_binary = df[df['sentiment'] != 'neutral']`

In [ ]:
# TODO: Filter out neutral tweets
df_binary = df[df['sentiment'] != 'neutral']

# TODO: Repeat Steps 5 and 6 using df_binary instead of df_sample
# Compare accuracy with the 3-class model above


### Exercise 2 — Scale Up

We sampled 5,000 tweets for classroom speed. Now train on the full ~45,000 tweets.
How does accuracy change? How much longer does it take?

Fill in the table:

| Sample size | Accuracy | Training time |
|---|---|---|
| 5,000 | ? | ? |
| 45,000 | ? | ? |

In [ ]:
import time

# TODO: Repeat Step 5 without the .sample(5000) — use the full df
# Wrap the .fit() call in time.time() to measure training time
# start = time.time()
# model_full.fit(...)
# end = time.time()
# print(f"Training time: {end - start:.1f} seconds")


---
## Key Takeaways

| Concept | What You Learned |
|---|---|
| **HuggingFace** | World's largest AI/dataset platform — one line loads 45,000 real tweets |
| **EDA** | Always explore data first; class imbalance is the norm, not the exception |
| **Tweet preprocessing** | Social media text needs special handling: URLs, mentions, hashtags |
| **TF-IDF** | Converts words to numbers weighted by importance and rarity |
| **Train/Test Split** | Never evaluate on training data — fundamental ML rule |
| **Naive Bayes** | Fast, interpretable, strong baseline for text classification |
| **Confusion Matrix** | Shows exactly where your model makes mistakes |

---

## Bonus — Going Further

**Level 2 — Pre-trained transformer (much more powerful):**
```python
from transformers import pipeline
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
classifier('This flight was absolutely incredible!')
```

**Level 3 — Real-time brand monitoring:**  
Connect to a social media API + this classifier to track sentiment about any company in real time.  
This is exactly what Lufthansa, Deutsche Bahn, and most major brands run today.

**Level 4 — Interactive dashboard:**  
Use `plotly Dash` to build a web interface where users type any tweet and instantly see the predicted sentiment and confidence breakdown.

---